In [1]:
import lightgbm as lgb

from lightgbm import LGBMClassifier

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    confusion_matrix
)
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
import optuna

In [2]:
train_df = pd.read_csv("training_bob(2003-2023).csv")
test_df = pd.read_csv("test_bob(2024-25).csv")

print(train_df.shape)
print(test_df.shape)

(3375208, 14)
(176241, 14)


In [3]:
FEATURES = [
    "CHLOR_A",
    "day_sin",
    "day_cos",
    "month_sin",
    "month_cos",
    "LAT_scaled",
    "LON_scaled"
]

TARGET = "PHYTOBLOOM"

In [4]:
X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_test = test_df[FEATURES]
y_test = test_df[TARGET]

In [5]:
lgb_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    random_state=42,
)

lgb_model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.028656 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 869
[LightGBM] [Info] Number of data points in the train set: 3375208, number of used features: 7
[LightGBM] [Info] Start training from score -0.039413
[LightGBM] [Info] Start training from score -4.385591
[LightGBM] [Info] Start training from score -3.642329


,objective,'multiclass'
,random_state,42
,num_class,3
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,class_weight,None
,min_split_gain,0.0


In [6]:
train_pred = lgb_model.predict(X_train)
test_pred = lgb_model.predict(X_test)

In [9]:
print("TRAIN RESULTS\n")

print(classification_report(y_train, train_pred))


print("Macro F1:",
      f1_score(y_train, train_pred, average='macro'))


print("\n\nTEST RESULTS\n")

print(classification_report(y_test, test_pred))


print("Macro F1:",
      f1_score(y_test, test_pred, average='macro'))

TRAIN RESULTS

              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99   3244767
         1.0       0.75      0.68      0.71     42040
         2.0       0.76      0.71      0.74     88401

    accuracy                           0.98   3375208
   macro avg       0.83      0.80      0.81   3375208
weighted avg       0.98      0.98      0.98   3375208

Macro F1: 0.8139887975606497


TEST RESULTS

              precision    recall  f1-score   support

         0.0       0.98      0.98      0.98    162353
         1.0       0.62      0.63      0.63      5693
         2.0       0.81      0.68      0.74      8195

    accuracy                           0.96    176241
   macro avg       0.80      0.76      0.78    176241
weighted avg       0.96      0.96      0.96    176241

Macro F1: 0.7813902758808188


In [8]:
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": lgb_model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print(importance)

      Feature  Importance
1     day_sin        2572
2     day_cos        1733
0     CHLOR_A        1595
6  LON_scaled        1508
5  LAT_scaled        1392
3   month_sin         115
4   month_cos          85


In [18]:
def objective_bob_v2(trial):
    params = {
        "objective": "multiclass",
        "num_class": 3,
        "metric": "multi_logloss",

        "n_estimators": trial.suggest_int("n_estimators", 20, 350, step=10),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.20, log=True),

        # Capped down from 13 — that depth is likely what caused the
        # CV-vs-final-fit mismatch
        "max_depth": trial.suggest_int("max_depth", 4, 14),

        # Capped down from 287 for the same reason
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),

        # Keep the low floor — minority classes still need small leaves
        # to get captured, just don't let overall depth run away
        "min_child_samples": trial.suggest_int("min_child_samples", 8, 50),

        "subsample": trial.suggest_float("subsample", 0.65, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.95),

        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
        "min_split_gain": trial.suggest_float("min_split_gain", 1e-3, 0.5, log=True),

        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,
    }

    weight_mode = trial.suggest_categorical("weight_mode", ["none", "balanced", "sqrt_balanced"])

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    gaps, val_scores = [], []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        if weight_mode == "none":
            sw = None
        elif weight_mode == "balanced":
            sw = compute_sample_weight(class_weight="balanced", y=y_tr)
        else:
            raw_w = compute_sample_weight(class_weight="balanced", y=y_tr)
            sw = np.sqrt(raw_w)

        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr, y_tr, sample_weight=sw)

        train_f1 = f1_score(y_tr, model.predict(X_tr), average="macro")
        val_f1 = f1_score(y_val, model.predict(X_val), average="macro")

        val_scores.append(val_f1)
        gaps.append(train_f1 - val_f1)

    mean_val = np.mean(val_scores)
    mean_gap = np.mean(gaps)

    return mean_val, mean_gap

In [19]:
study_bob_v2 = optuna.create_study(directions=["maximize", "minimize"]) 
study_bob_v2.optimize(objective_bob_v2, n_trials=60, show_progress_bar=True)

[I 2026-07-14 14:38:12,158] A new study created in memory with name: no-name-16c7b83c-c851-4cfe-b1e0-e0601c0fcce6


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-07-14 14:46:24,086] Trial 0 finished with values: [0.8533482917549421, 0.017028266962043247] and parameters: {'n_estimators': 330, 'learning_rate': 0.09988910097235677, 'max_depth': 7, 'num_leaves': 59, 'min_child_samples': 18, 'subsample': 0.7178242525334307, 'colsample_bytree': 0.9116469729441918, 'reg_alpha': 0.0022129743276913193, 'reg_lambda': 0.002964977716837036, 'min_split_gain': 0.11241971836808402, 'weight_mode': 'none'}.
[I 2026-07-14 14:48:16,518] Trial 1 finished with values: [0.7805895300634853, 0.003673568006721628] and parameters: {'n_estimators': 80, 'learning_rate': 0.056417694548823824, 'max_depth': 11, 'num_leaves': 105, 'min_child_samples': 46, 'subsample': 0.7640941397728092, 'colsample_bytree': 0.8440073432067517, 'reg_alpha': 0.0046363694218727325, 'reg_lambda': 0.0345381352435878, 'min_split_gain': 0.017107389029571513, 'weight_mode': 'sqrt_balanced'}.
[I 2026-07-14 14:52:51,553] Trial 2 finished with values: [0.7585572865532966, 0.0049838259867432955] 

In [21]:
best_trials = study_bob_v2.best_trials
for t in best_trials:
    print(f"macro_f1={t.values[0]:.4f}, gap={t.values[1]:.4f}, params={t.params}")

macro_f1=0.7998, gap=0.0038, params={'n_estimators': 90, 'learning_rate': 0.08141307820882503, 'max_depth': 13, 'num_leaves': 24, 'min_child_samples': 38, 'subsample': 0.8412126102880177, 'colsample_bytree': 0.8063603564525543, 'reg_alpha': 0.0029569674058079073, 'reg_lambda': 4.701838054983387, 'min_split_gain': 0.049685903754591754, 'weight_mode': 'none'}
macro_f1=0.7762, gap=0.0028, params={'n_estimators': 230, 'learning_rate': 0.0440236960337349, 'max_depth': 8, 'num_leaves': 42, 'min_child_samples': 22, 'subsample': 0.7021618881755594, 'colsample_bytree': 0.8283211912283841, 'reg_alpha': 0.272264996114518, 'reg_lambda': 1.1172374640807676, 'min_split_gain': 0.09951164175773052, 'weight_mode': 'sqrt_balanced'}
macro_f1=0.8537, gap=0.0156, params={'n_estimators': 320, 'learning_rate': 0.10444621551959582, 'max_depth': 7, 'num_leaves': 98, 'min_child_samples': 47, 'subsample': 0.727276215944423, 'colsample_bytree': 0.8950516587584456, 'reg_alpha': 0.14867848905463754, 'reg_lambda': 4

In [22]:
i=0
for t in best_trials:
    params=t.params
    lgb_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    **params,
    random_state=42
    )

    lgb_model.fit(X_train, y_train)
    y_pred1 = lgb_model.predict(
        X_train
    )

    y_pred = lgb_model.predict(
        X_test
    )
    macro_f1_train = f1_score(
        y_train,
        y_pred1,
        average="macro"
    )

    macro_f1_test = f1_score(
        y_test,
        y_pred,
        average="macro"
    )
    gap=macro_f1_train-macro_f1_test
    print(f"{i} Gap: {gap} Train Macro F1 : {macro_f1_train}  Test Macro F1 : {macro_f1_test}")
    i+=1


0 Gap: 0.02173105632034622 Train Macro F1 : 0.8033541634841521  Test Macro F1 : 0.7816231071638059
1 Gap: 0.048882447661439454 Train Macro F1 : 0.8250347284933689  Test Macro F1 : 0.7761522808319294
2 Gap: 0.08913488696969696 Train Macro F1 : 0.8681355030386086  Test Macro F1 : 0.7790006160689117
3 Gap: -0.02321441508455868 Train Macro F1 : 0.7677844388076829  Test Macro F1 : 0.7909988538922416
4 Gap: 0.051889433161432574 Train Macro F1 : 0.8194576753083419  Test Macro F1 : 0.7675682421469093
5 Gap: -0.020145796423343132 Train Macro F1 : 0.7669745431660022  Test Macro F1 : 0.7871203395893454
6 Gap: 0.008117974955474616 Train Macro F1 : 0.32777908334290423  Test Macro F1 : 0.3196611083874296
7 Gap: -0.02095820767737777 Train Macro F1 : 0.7717523394929001  Test Macro F1 : 0.7927105471702779
8 Gap: 0.05438321854975836 Train Macro F1 : 0.8094391825408239  Test Macro F1 : 0.7550559639910656
9 Gap: -0.01617366212048621 Train Macro F1 : 0.7752447301628923  Test Macro F1 : 0.7914183922833785
1

In [25]:
params=best_trials[10].params
params

{'n_estimators': 320,
 'learning_rate': 0.025754508160065227,
 'max_depth': 5,
 'num_leaves': 65,
 'min_child_samples': 46,
 'subsample': 0.6815372336035705,
 'colsample_bytree': 0.937585105687456,
 'reg_alpha': 0.24712644463561262,
 'reg_lambda': 0.047393210901825036,
 'min_split_gain': 0.47128768439445706,
 'weight_mode': 'none'}

In [26]:
params

{'n_estimators': 320,
 'learning_rate': 0.025754508160065227,
 'max_depth': 5,
 'num_leaves': 65,
 'min_child_samples': 46,
 'subsample': 0.6815372336035705,
 'colsample_bytree': 0.937585105687456,
 'reg_alpha': 0.24712644463561262,
 'reg_lambda': 0.047393210901825036,
 'min_split_gain': 0.47128768439445706,
 'weight_mode': 'none'}

In [27]:
lgb_model = LGBMClassifier(
    objective='multiclass',
    num_class=3,
    **params,
    random_state=42
    )

lgb_model.fit(X_train, y_train)
y_pred1 = lgb_model.predict(
    X_train
)

y_pred = lgb_model.predict(
    X_test
)
macro_f1_train = f1_score(
    y_train,
    y_pred1,
    average="macro"
)

macro_f1_test = f1_score(
    y_test,
    y_pred,
    average="macro"
)
gap=macro_f1_train-macro_f1_test
print(f" Gap: {gap} Train Macro F1 : {macro_f1_train}  Test Macro F1 : {macro_f1_test}")

 Gap: 0.0030617730702492407 Train Macro F1 : 0.7907327666055178  Test Macro F1 : 0.7876709935352686


In [28]:
import joblib

joblib.dump(lgb_model, 'lgb_model_BOB.pkl')

['lgb_model_BOB.pkl']